# MiniGrid Benchmark Runner
Notebook para executar benchmark_minigrid em ambiente local, Colab ou Kaggle.

## 1 - Instalações e Configurações Diversas

Basta executar tudo desta seção. Não precisa alterar nada.

In [1]:
try:
    import kaggle_secrets
    EXECUTION_ENV = "kaggle"
except ImportError:
    try:
        import google.colab
        EXECUTION_ENV = "colab"
    except ImportError:
        EXECUTION_ENV = "local"

print(f"Execution environment: {EXECUTION_ENV}")

Execution environment: local


In [2]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q minigrid==3.1.0
    !pip install -q langchain-openai>=0.0.1
    !pip install -q langchain-deepseek>=0.0.1
    !pip install -q langchain-huggingface==1.2.2
    !pip install -q transformers>=5.5.4
    !pip install -U bitsandbytes==0.49.2

In [3]:
import sys
import os

if EXECUTION_ENV == "colab":
    repo_path = os.path.join(os.getcwd(), "minigrid_benchmark")
elif EXECUTION_ENV == "kaggle":
    repo_path = "/kaggle/working/minigrid_benchmark"
else:
    cwd = os.getcwd()
    candidates = [cwd, os.path.abspath(os.path.join(cwd, ".."))]
    repo_path = None
    for candidate in candidates:
        if os.path.exists(os.path.join(candidate, "src", "benchmark_minigrid.py")):
            repo_path = candidate
            break
    if repo_path is None:
        repo_path = cwd

if EXECUTION_ENV in ["colab", "kaggle"] and not os.path.exists(repo_path):
    !git clone https://github.com/pablo-sampaio/minigrid_benchmark.git {repo_path}

src_path = os.path.join(repo_path, "src")
sys.path.append(src_path)

print(f"repo_path={repo_path}")
print(f"src_path={src_path}")

repo_path=d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark
src_path=d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\src


In [4]:
import experiments_util
from benchmark_minigrid import run_benchmark_minigrid

c:\.venvs\langchain\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [5]:
if EXECUTION_ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    results_dir = '/content/drive/My Drive/EAD-Pesquisa-Agentes/results'
elif EXECUTION_ENV == "kaggle":
    results_dir = '/kaggle/working/results'
else:
    results_dir = os.path.abspath(os.path.join(repo_path, 'results'))

os.makedirs(results_dir, exist_ok=True)
experiments_util.RESULTS_BASE_DIR = results_dir
print(f"results_dir={results_dir}")

results_dir=d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results


In [6]:
def resolve_api_key(provider):
    api_key = None

    if provider == "openai":
        secret_name = "OPENAI_API_KEY" 
    elif provider == "deepseek":
        secret_name = "DEEPSEEK_API_KEY"
    elif provider == "hf":
        secret_name = "HF_TOKEN"
    else:
        print("Invalid provider: ", provider)
        return None

    if EXECUTION_ENV == "colab":
        from google.colab import userdata
        api_key = userdata.get(secret_name)
    elif EXECUTION_ENV == "kaggle":
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        api_key = user_secrets.get_secret(secret_name)
    else:
        api_key = os.getenv(secret_name)
    
    return api_key

## 2 - Defina Aqui o Modelo

In [7]:
# dictionary provider -> model options, quantization
MODEL_OPTIONS = {
    "openai": [ ("gpt-5.4", None), ("gpt-5.4-mini", None) ],
    "deepseek": [
        ("deepseek-v4-flash", None),
        ("deepseek-v4-pro", None)
    ],
    "hf": [
        ("google/gemma-4-E2B-it", None), 
        ("google/gemma-4-12B-it-qat-q4_0-gguf", None),
        ("WeiboAI/VibeThinker-3B", "8bit")
    ]
}

In [8]:
import ipywidgets as widgets
from IPython.display import display

PROVIDER = list(MODEL_OPTIONS.keys())[0]
MODEL_ID, QUANTIZATION = MODEL_OPTIONS[PROVIDER][0]

provider_dd = widgets.Dropdown(options=list(MODEL_OPTIONS.keys()), value=PROVIDER, description="Provider:")

model_dd = widgets.Dropdown(description="Model:")  #layout=widgets.Layout(width="900px"))

def update_model_options(*_):
    global PROVIDER, MODEL_ID, QUANTIZATION
    PROVIDER = provider_dd.value
    model_dd.options = [
        (f"{model_id} | quantization={quantization}", (model_id, quantization))
        for model_id, quantization in MODEL_OPTIONS[PROVIDER]
    ]
    model_dd.value = model_dd.options[0][1]
    MODEL_ID, QUANTIZATION = model_dd.value

def update_model_value(change):
    global MODEL_ID, QUANTIZATION
    MODEL_ID, QUANTIZATION = change["new"]

provider_dd.observe(update_model_options, names="value")
model_dd.observe(update_model_value, names="value")

update_model_options()
display(provider_dd, model_dd)

Dropdown(description='Provider:', options=('openai', 'deepseek', 'hf'), value='openai')

Dropdown(description='Model:', options=(('gpt-5.4 | quantization=None', ('gpt-5.4', None)), ('gpt-5.4-mini | q…

In [9]:
#results_folder_name = None

# caso queria continuar, indicar o nome da pasta
results_folder_name = 'benchmark_openai_gpt-5.4_2026-06-20_08h27min'

## 3 - Execução dos Testes do Benchark

In [10]:
API_KEY = resolve_api_key(PROVIDER)

In [11]:
final_results, summary_path = run_benchmark_minigrid(
    provider=PROVIDER,
    model_id=MODEL_ID,
    api_key=API_KEY,
    results_base_dir=results_dir,
    results_folder_name=results_folder_name,
    quantization=QUANTIZATION if PROVIDER == "hf" else None,
    verbose=True,
)

print('Saved summary to:', summary_path)

Resuming from: d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results\benchmark_openai_gpt-5.4_2026-06-20_08h27min\summary.json
Results will be saved to: d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results\benchmark_openai_gpt-5.4_2026-06-20_08h27min\summary.json


Completed Experiment Configurations:   0%|          | 0/8 [00:00<?, ?it/s]

Completed Environments:   0%|          | 0/2 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

...

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Completed Environments:   0%|          | 0/2 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Saved summary to: d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results\benchmark_openai_gpt-5.4_2026-06-20_08h27min\summary.json


In [12]:
print("Arquivo JSON com o sumário dos resultados:", summary_path)

Arquivo JSON com o sumário dos resultados: d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results\benchmark_openai_gpt-5.4_2026-06-20_08h27min\summary.json


In [13]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    # zip the final results directory
    import shutil

    benchmark_result_dir = os.path.dirname(summary_path)
    benchrmark_name = os.path.basename(benchmark_result_dir)
    zip_path = os.path.join(os.path.dirname(benchmark_result_dir), f"{benchrmark_name}_results_zip")

    # Creates results_zip.zip containing everything
    shutil.make_archive(zip_path, 'zip', benchmark_result_dir)

    print("Arquivo compactado com todos os resultados:", f"{zip_path}.zip")